# Day 5 — Delta DML: MERGE, UPDATE, DELETE, SCD patterns

Plan for about 1.5–2 hours. You will build a small `dim_routes` table on ABFS, merge updates from the Python API and from a staging Delta path, run conditional merges, use UPDATE and DELETE (including no-op examples), review SCD Type 1 vs a simplified Type 2 layout, and try an optional SQL MERGE plus follow-up challenges.

SQL uses `delta.` paths where shown. Earlier notebooks in this course use the same ABFS layout and mount helper.

In [0]:
from pyspark.sql.functions import col, concat_ws, current_date, current_timestamp, lit, sha2, when

BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u01"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_DIM = DAY5 + "/delta/dim_routes"
P_SCD2 = DAY5 + "/delta/dim_routes_scd2"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"

# Verify source CSV exists
print("=== Checking Prerequisites ===")
try:
    df_csv = spark.read.format("csv").option("header", True).option("inferSchema", True).load(SOURCE_CSV).limit(1)
    print(f"✓ Source CSV exists: {SOURCE_CSV}")
except Exception as e:
    print(f"✗ Source CSV NOT found: {SOURCE_CSV}")
    print(f"  Error: {type(e).__name__}")
    print(f"  Hint: Run notebook 02 mount first, then check source data on ADLS")

print(f"\nDIM tables will be created in:")
print(f"  P_DIM (Type 1):  {P_DIM}")
print(f"  P_SCD2 (Type 2): {P_SCD2}")

## S1 — Build **`dim_routes`** (surrogate key = `route_key`)

In [0]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit

src = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count")
    .withColumnRenamed("count", "flight_count")
    .withColumn("route_key", sha2(concat_ws("|", col("DEST_COUNTRY_NAME"), col("ORIGIN_COUNTRY_NAME")), 256))
    .withColumn("effective_from", current_date())
    .withColumn("updated_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
)
src.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_DIM)
src.display(5, truncate=False)
print("rows", src.count())

## S2 — **MERGE** — apply corrections from staging

In [0]:
from delta.tables import DeltaTable

dim = DeltaTable.forPath(spark, P_DIM)

updates = (
    spark.read.format("delta")
    .load(P_DIM)
    .filter(col("DEST_COUNTRY_NAME") == "United States")
    .limit(5)
    .withColumn("flight_count", col("flight_count") + lit(100))
    .withColumn("updated_at", current_timestamp())
)

(
    dim.alias("t")
    .merge(updates.alias("s"), "t.route_key = s.route_key")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DeltaTable.forPath(spark, P_DIM).history(5).select("version", "operation", "operationMetrics").display(truncate=False)

## S2b — **Conditional** `whenMatchedUpdate` (only if staging is “newer”)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit

dim = DeltaTable.forPath(spark, P_DIM)
bump = (
    spark.read.format("delta")
    .load(P_DIM)
    .where("DEST_COUNTRY_NAME = 'Germany'")
    .limit(4)
    .withColumn("flight_count", col("flight_count") + lit(50))
    .withColumn("updated_at", current_timestamp())
)
(
    dim.alias("t")
    .merge(bump.alias("s"), "t.route_key = s.route_key")
    .whenMatchedUpdate(
        condition="s.flight_count > t.flight_count",
        set={"flight_count": "s.flight_count", "updated_at": "s.updated_at"},
    )
    .execute()
)
print("Conditional merge applied where staged flight_count exceeds target")

## S2c — **Stage → merge** (separate Delta path, full refresh pattern)

In [0]:
from pyspark.sql.functions import coalesce, col, concat_ws, current_date, current_timestamp, lit, sha2

P_STAGE = DAY5 + "/delta/stage_routes_snapshot"
st = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count")
    .withColumnRenamed("count", "flight_count")
    .withColumn("route_key", sha2(concat_ws("|", col("DEST_COUNTRY_NAME"), col("ORIGIN_COUNTRY_NAME")), 256))
    .withColumn("effective_from", current_date())
    .withColumn("updated_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
    .limit(200)
)
st.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_STAGE)
print("Staging rows:", st.count(), "at", P_STAGE)

In [0]:
from delta.tables import DeltaTable

dim = DeltaTable.forPath(spark, P_DIM)
stg = spark.read.format("delta").load(P_STAGE)
(
    dim.alias("t")
    .merge(stg.alias("s"), "t.route_key = s.route_key")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("Merged staging snapshot into dim_routes")

## S3 — Verify merge on touched **route_key**s

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

keys = [r["route_key"] for r in spark.read.format("delta").load(P_DIM).where("DEST_COUNTRY_NAME = 'United States'").limit(5).select("route_key").collect()]
ver = int(DeltaTable.forPath(spark, P_DIM).history(1).select("version").collect()[0]["version"])
print("latest version", ver)
if ver >= 1:
    spark.read.format("delta").option("versionAsOf", ver - 1).load(P_DIM).filter(col("route_key").isin(keys)).select("route_key", "flight_count").orderBy("route_key").display(truncate=False)
spark.read.format("delta").load(P_DIM).filter(col("route_key").isin(keys)).select("route_key", "flight_count").orderBy("route_key").display(truncate=False)

## S4 — **`UPDATE`** (DeltaTable API — avoids SQL dialect quirks)

In [0]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, P_DIM).update(
    condition="DEST_COUNTRY_NAME = 'Canada'",
    set={"flight_count": "flight_count + 1"},
)
print("Updated Canada rows (+1 flight_count each)")

## S5 — **`DELETE`** (SQL + **DeltaTable** API)

In [0]:
# Safe no-op: predicate matches nothing
spark.sql(f"DELETE FROM delta.`{P_DIM}` WHERE DEST_COUNTRY_NAME = '__no_such_country__'")
print("SQL delete completed (0 rows if predicate empty)")

In [0]:
from delta.tables import DeltaTable

# API delete — same semantics as SQL DELETE with predicate
DeltaTable.forPath(spark, P_DIM).delete("DEST_COUNTRY_NAME = '__also_missing__'")
print("DeltaTable.delete no-op demo OK")